# Prompt-based sentence classification

Compare context settings and prompt levels on the sentence reference and category tasks.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Performance

In [ ]:
from prompting.evaluation import evaluate_multilabel_binary, evaluate_sentence_categories

binary_scores = evaluate_multilabel_binary(paths.PROMPT_RESULTS / 'sentence')
category_scores, averages = evaluate_sentence_categories(paths.PROMPT_RESULTS / 'sentence')


## BERT on the same sentences

In [ ]:
from prompting.bert_comparison import run

run()


## Prompt and model comparison

In [ ]:
from prompting.comparison_plots import main

main()


## Category transitions

In [ ]:
import os
import ast
import pandas as pd

BASE_DIR = str(paths.PROMPT_RESULTS / 'sentence')
CONTEXT_DIRS = {'nocontext': os.path.join(BASE_DIR, 'nocontext'), 'context_4': os.path.join(BASE_DIR, 'context_4')}
CONTEXT_ORDER = {'nocontext': 0, 'context_4': 1}
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
TRANSITIONS = list(zip(LEVELS[:-1], LEVELS[1:]))
TRANSITION_LABELS = [f'{a}→{b}' for a, b in TRANSITIONS]
TRANSITION_ORDER = {lab: i for i, lab in enumerate(TRANSITION_LABELS)}
CSV_PATTERN = 'llm_results_sentence_multilabel_{level}.csv'
COL_LETTER_ID = 'letter_id'
COL_SENTENCE_ID = 'sentence_id'
COL_GOLD = 'gold_categories'
COL_PRED = 'predicted_tokens'
CATEGORIES = list('ABCDEFGHI')
CATEGORY_NAMES = {
    'A': 'Inspirations and Influences',
    'B': 'Writing Process and Creative Insight',
    'C': 'Elements of Fictional Craft',
    'D': 'Editorial and Publishing Processes',
    'E': 'Quality and Evaluation',
    'F': 'Financial and Legal Aspects',
    'G': 'Gratitude and Acknowledgement',
    'H': 'Marketing and Promotion',
    'I': 'Impact and Reception',
    'J': 'No reference to Dickens’s works present',
}
OUT_COMBINED = os.path.join(BASE_DIR, 'category_transition_summary_all_contexts.csv')
OUT_TRANSITION_PATTERN = 'category_transition_{transition}_both_contexts.csv'
from prompting.analysis_utils import read_file
from prompting.analysis_utils import parse_categories as parse_label_list


def pick_id_columns(df: pd.DataFrame) -> list:
    missing = [c for c in [COL_LETTER_ID, COL_SENTENCE_ID] if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required ID columns for sentence-level merge: {missing}')
    return [COL_LETTER_ID, COL_SENTENCE_ID]
from prompting.analysis_utils import compute_confusion


def safe_precision(tp: int, fp: int) -> float:
    denom = tp + fp
    if denom == 0:
        return 0.0
    return tp / denom
from prompting.analysis_utils import safe_recall


def f1_from_counts(tp: int, fp: int, fn: int) -> float:
    denom = 2 * tp + fp + fn
    if denom == 0:
        return 0.0
    return 2 * tp / denom


def build_level_df(path: str) -> tuple[pd.DataFrame, list]:
    df = read_file(path)
    missing = [c for c in [COL_LETTER_ID, COL_SENTENCE_ID, COL_GOLD, COL_PRED] if c not in df.columns]
    if missing:
        raise ValueError(f'File {os.path.basename(path)} missing columns: {missing}')
    id_cols = pick_id_columns(df)
    df['_gold_set'] = df[COL_GOLD].apply(parse_label_list)
    df['_pred_set'] = df[COL_PRED].apply(parse_label_list)
    return (df, id_cols)


def merge_levels(df1: pd.DataFrame, df2: pd.DataFrame, id_cols: list) -> pd.DataFrame:
    merged = pd.merge(
        df1[id_cols + ['_gold_set', '_pred_set']].rename(columns={'_pred_set': '_pred_set_1'}),
        df2[id_cols + ['_gold_set', '_pred_set']].rename(
            columns={'_pred_set': '_pred_set_2', '_gold_set': '_gold_set_2'},
        ),
        on=id_cols,
        how='inner',
    )
    mismatches = int(merged['_gold_set'].ne(merged['_gold_set_2']).sum())
    if mismatches > 0:
        print(
            f' WARNING: {mismatches} instances have differing gold label sets between files (unexpected).',
        )
    merged = merged.drop(columns=['_gold_set_2'])
    return merged


def per_category_transition_table(
    merged: pd.DataFrame,
    level_a: str,
    level_b: str,
    context_label: str,
) -> pd.DataFrame:
    rows = []
    gold_sets = merged['_gold_set']
    pred_a = merged['_pred_set_1']
    pred_b = merged['_pred_set_2']
    transition_label = f'{level_a}→{level_b}'
    for cat in CATEGORIES:
        cat_name = CATEGORY_NAMES.get(cat, cat)
        gold_has = gold_sets.apply(lambda s: cat in s)
        pred_a_has = pred_a.apply(lambda s: cat in s)
        pred_b_has = pred_b.apply(lambda s: cat in s)
        conf_a = compute_confusion(gold_has, pred_a_has)
        conf_b = compute_confusion(gold_has, pred_b_has)
        fn_to_tp = int((gold_has & ~pred_a_has & pred_b_has).sum())
        tp_to_fn = int((gold_has & pred_a_has & ~pred_b_has).sum())
        tn_to_fp = int((~gold_has & ~pred_a_has & pred_b_has).sum())
        fp_to_tn = int((~gold_has & pred_a_has & ~pred_b_has).sum())
        p_a = safe_precision(conf_a['TP'], conf_a['FP'])
        r_a = safe_recall(conf_a['TP'], conf_a['FN'])
        f1_a = f1_from_counts(conf_a['TP'], conf_a['FP'], conf_a['FN'])
        p_b = safe_precision(conf_b['TP'], conf_b['FP'])
        r_b = safe_recall(conf_b['TP'], conf_b['FN'])
        f1_b = f1_from_counts(conf_b['TP'], conf_b['FP'], conf_b['FN'])
        rows.append(
            {
                'transition': transition_label,
                'transition_order': TRANSITION_ORDER[transition_label],
                'context': context_label,
                'context_order': CONTEXT_ORDER.get(context_label, 999),
                'category': cat_name,
                '_category_code': cat,
                f'TP_{level_a}': conf_a['TP'],
                f'FP_{level_a}': conf_a['FP'],
                f'FN_{level_a}': conf_a['FN'],
                f'TN_{level_a}': conf_a['TN'],
                f'F1_{level_a}': round(f1_a, 4),
                f'TP_{level_b}': conf_b['TP'],
                f'FP_{level_b}': conf_b['FP'],
                f'FN_{level_b}': conf_b['FN'],
                f'TN_{level_b}': conf_b['TN'],
                f'F1_{level_b}': round(f1_b, 4),
                'FN→TP': fn_to_tp,
                'TP→FN': tp_to_fn,
                'TN→FP': tn_to_fp,
                'FP→TN': fp_to_tn,
                'ΔF1': round(f1_b - f1_a, 4),
                'ΔP': round(p_b - p_a, 4),
                'ΔR': round(r_b - r_a, 4),
            },
        )
    return pd.DataFrame(rows)


def print_transition_summary(df_transition: pd.DataFrame, transition_label: str) -> None:
    print(f'\n=== TRANSITION {transition_label} ===')
    for context_label, sub in df_transition.groupby('context', sort=False):
        sub = sub.sort_values('_category_code')
        cols = ['category', 'ΔP', 'ΔR', 'ΔF1', 'FN→TP', 'TP→FN', 'TN→FP', 'FP→TN']
        top_gain = sub.sort_values('ΔF1', ascending=False).head(3)[cols]
        top_drop = sub.sort_values('ΔF1', ascending=True).head(3)[cols]
        print(f'\n-- {context_label} --')
        print('Top ΔF1 gains:')
        print(top_gain.to_string(index=False))
        print('Top ΔF1 drops:')
        print(top_drop.to_string(index=False))


def main():
    all_tables = []
    for context_label, context_dir in CONTEXT_DIRS.items():
        if not os.path.isdir(context_dir):
            print(f' Skipping missing directory: {context_dir}')
            continue
        level_dfs = {}
        id_cols_ref = None
        for lvl in LEVELS:
            path = os.path.join(context_dir, CSV_PATTERN.format(level=lvl))
            if not os.path.isfile(path):
                raise FileNotFoundError(f'Missing expected file: {path}')
            df, id_cols = build_level_df(path)
            level_dfs[lvl] = df
            if id_cols_ref is None:
                id_cols_ref = id_cols
        for a, b in TRANSITIONS:
            merged = merge_levels(level_dfs[a], level_dfs[b], id_cols_ref)
            if merged.empty:
                print(
                    f' Empty merge for {context_label} {a}→{b}. Check (letter_id, sentence_id) alignment.',
                )
                continue
            out_tbl = per_category_transition_table(merged, a, b, context_label)
            all_tables.append(out_tbl)
    if not all_tables:
        print('\nNo outputs produced (no context directories processed).')
        return
    combined = pd.concat(all_tables, ignore_index=True)
    combined = combined.sort_values(
        ['transition_order', 'context_order', '_category_code'],
        ascending=[True, True, True],
    )
    for transition_label in TRANSITION_LABELS:
        df_t = combined[combined['transition'] == transition_label].copy()
        if df_t.empty:
            continue
        df_t = df_t.sort_values(['context_order', '_category_code'])
        print_transition_summary(df_t, transition_label)
    for transition_label in TRANSITION_LABELS:
        df_t = combined[combined['transition'] == transition_label].copy()
        if df_t.empty:
            continue
        df_t_out = df_t.drop(columns=['_category_code'])
        out_name = OUT_TRANSITION_PATTERN.format(transition=transition_label.replace('→', '_to_'))
        out_path = os.path.join(BASE_DIR, out_name)
        df_t_out.to_csv(out_path, index=False)
        print(f'\n Saved per-transition (both contexts): {out_path}')
    combined_out = combined.drop(columns=['_category_code'])
    combined_out.to_csv(OUT_COMBINED, index=False)
    print(f'\n Saved combined summary across contexts/transitions: {OUT_COMBINED}')
if __name__ == '__main__':
    main()


## Transition plots

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = str(paths.PROMPT_RESULTS / 'sentence')
INFILE = os.path.join(BASE_DIR, 'category_transition_summary_all_contexts.csv')
EXPORT_DIR = str(paths.FIGURES_DIR)
CONTEXTS = ['nocontext', 'context_4']
TRANSITIONS = ['L1→L2', 'L2→L3', 'L3→L4', 'L4→L5']
TOPN = 3
POS_COLOR = 'lightcoral'
NEG_COLOR = '#4682B4'


def normalise_transition(s: str) -> str:
    s = str(s).strip().replace('->', '→').replace('=>', '→')
    s = re.sub('\\s+', '', s)
    return s


def transition_slug(trans: str) -> str:
    return normalise_transition(trans).replace('→', '_to_')


def load_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df['transition'] = df['transition'].apply(normalise_transition)
    df['ΔF1'] = pd.to_numeric(df['ΔF1'], errors='coerce')
    return df


def top_bottom_sorted(sub: pd.DataFrame, n: int=TOPN) -> pd.DataFrame:
    sub = sub.dropna(subset=['ΔF1']).copy()
    top = sub.sort_values('ΔF1', ascending=False).head(n).copy()
    bot = sub.sort_values('ΔF1', ascending=True).head(n).copy()
    top = top.sort_values('ΔF1', ascending=False)
    bot = bot.sort_values('ΔF1', ascending=False)
    return pd.concat([top, bot], ignore_index=True)


def wrap_labels(labels, width=42):
    out = []
    for s in labels:
        words = str(s).split()
        line, lines = ('', [])
        for w in words:
            cand = (line + ' ' + w).strip()
            if len(cand) <= width:
                line = cand
            else:
                if line:
                    lines.append(line)
                line = w
        if line:
            lines.append(line)
        out.append('\n'.join(lines))
    return out


def plot_transition(df: pd.DataFrame, trans: str, xlim: float, export_dir: str):
    plt.rcParams.update(
        {
            'font.size': 15,
            'axes.titlesize': 16,
            'axes.labelsize': 16,
            'xtick.labelsize': 15,
            'ytick.labelsize': 15,
            'pdf.fonttype': 42,
            'ps.fonttype': 42,
        },
    )
    CTX_TITLE = {'nocontext': 'No context', 'context_4': 'With context (n=4)'}
    sub_all = df[df['transition'] == trans].copy()
    if sub_all.empty:
        print(f' No data for transition {trans}')
        return
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13.2, 5.2), sharex=True, constrained_layout=True)
    xlabel = f'ΔF1 ({trans})'
    for j, ctx in enumerate(CONTEXTS):
        ax = axes[j]
        sub = sub_all[sub_all['context'] == ctx].copy()
        tb = top_bottom_sorted(sub, TOPN)
        names = wrap_labels(tb['category'].astype(str).tolist(), width=42)
        vals = tb['ΔF1'].astype(float).values
        colors = [POS_COLOR if v >= 0 else NEG_COLOR for v in vals]
        y = np.arange(len(names))
        ax.barh(y, vals, color=colors)
        ax.axvline(0.0, linewidth=1.2, color='black')
        ax.grid(True, axis='x', alpha=0.25)
        ax.set_xlim(-xlim, xlim)
        ax.set_yticks(y)
        ax.set_yticklabels(names)
        ax.invert_yaxis()
        ax.axhline(2.5, linewidth=1.2, alpha=0.85, color='black')
        ax.set_title(CTX_TITLE.get(ctx, ctx), pad=8)
        ax.set_xlabel(xlabel, labelpad=8)
        dx = 0.012 * xlim
        for yi, v in zip(y, vals):
            if v >= 0:
                ax.text(v + dx, yi, f'{v:+.3f}', va='center', ha='left')
            else:
                ax.text(v - dx, yi, f'{v:+.3f}', va='center', ha='right')
        ax.tick_params(axis='y', pad=8)
    os.makedirs(export_dir, exist_ok=True)
    slug = transition_slug(trans)
    png_path = os.path.join(export_dir, f'exp3_transition_{slug}_deltaF1.png')
    fig.savefig(png_path, dpi=600, bbox_inches='tight')
    print(f' Saved: {png_path}')
    plt.close(fig)


def main():
    df = load_df(INFILE)
    df = df[df['context'].isin(CONTEXTS)].copy()
    df = df[df['transition'].isin(TRANSITIONS)].copy()
    finite = df['ΔF1'].dropna().values
    max_abs = float(np.max(np.abs(finite))) if len(finite) else 0.1
    global_xlim = max_abs * 1.15
    for trans in TRANSITIONS:
        plot_transition(df, trans, xlim=global_xlim, export_dir=EXPORT_DIR)
if __name__ == '__main__':
    main()


## Changed category assignments

In [ ]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from shared.utils import load_pickle

BASE_DIR = str(paths.PROMPT_RESULTS / 'sentence')
CONTEXT_DIRS = {'nocontext': os.path.join(BASE_DIR, 'nocontext'), 'context_4': os.path.join(BASE_DIR, 'context_4')}
CONTEXT_ORDER = {'nocontext': 0, 'context_4': 1}
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
TRANSITIONS = list(zip(LEVELS[:-1], LEVELS[1:]))
TRANSITION_LABELS = [f'{a}→{b}' for a, b in TRANSITIONS]
TRANSITION_ORDER = {lab: i for i, lab in enumerate(TRANSITION_LABELS)}
CSV_PATTERN = 'llm_results_sentence_multilabel_{level}.csv'
COL_LETTER_ID = 'letter_id'
COL_SENTENCE_ID = 'sentence_id'
COL_GOLD = 'gold_categories'
COL_PRED = 'predicted_tokens'
COL_PROBS_JSON = 'slot_probs_json'
TRACE_COLS = ['batch_id', 'valid', 'warnings', 'processing_time_sec', 'rationale', 'evidence']
CATEGORIES = list('ABCDEFGHI')
CATEGORY_NAMES = {
    'A': 'Inspirations and Influences',
    'B': 'Writing Process and Creative Insight',
    'C': 'Elements of Fictional Craft',
    'D': 'Editorial and Publishing Processes',
    'E': 'Quality and Evaluation',
    'F': 'Financial and Legal Aspects',
    'G': 'Gratitude and Acknowledgement',
    'H': 'Marketing and Promotion',
    'I': 'Impact and Reception',
    'J': 'No reference to Dickens’s works present',
}
CATEGORY_CODE_BY_NAME = {v: k for k, v in CATEGORY_NAMES.items() if k in CATEGORIES}
EXPORT_DIR = os.path.join(BASE_DIR, 'transition_shift_instances')
OUT_TRANSITION_PATTERN = 'transition_shift_instances_{transition}_both_contexts.csv'
OUT_COMBINED = os.path.join(EXPORT_DIR, 'transition_shift_instances_all.csv')
TOPK_OTHER_DELTA_PROBS = 6
SHOW_NONE_BUCKET = True
SUBCAT_NONE = '(none)'
SUBCAT_GOLD_NEG = '(gold negative)'
TOP_SUBCATS_ABS = 12
TOP_SUBCATS_OVERVIEW = 12
from prompting.analysis_utils import read_file
from prompting.analysis_utils import parse_categories as parse_label_list


def parse_probs_dict(x) -> dict:
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    s = str(x).strip()
    if not s:
        return {}
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    return {}
from prompting.analysis_utils import set_to_str


def delta_probs_summary(
    probs_a: dict,
    probs_b: dict,
    exclude_cat: str,
    k: int=TOPK_OTHER_DELTA_PROBS,
) -> str:
    pairs = []
    for c in CATEGORIES:
        if c == exclude_cat:
            continue
        try:
            pa = float(probs_a.get(c, np.nan))
            pb = float(probs_b.get(c, np.nan))
        except Exception:
            continue
        if np.isnan(pa) or np.isnan(pb):
            continue
        pairs.append((c, pb - pa))
    if not pairs:
        return ''
    pairs = sorted(pairs, key=lambda t: abs(t[1]), reverse=True)[:k]
    return '; '.join([f'{c}:{d:+.3f}' for c, d in pairs])
from prompting.analysis_utils import compute_confusion


def safe_precision(tp: int, fp: int) -> float:
    denom = tp + fp
    return tp / denom if denom else 0.0


def safe_recall(tp: int, fn: int) -> float:
    denom = tp + fn
    return tp / denom if denom else 0.0


def f1_from_counts(tp: int, fp: int, fn: int) -> float:
    denom = 2 * tp + fp + fn
    return 2 * tp / denom if denom else 0.0


def sign_arrow(x: float) -> str:
    if x > 0:
        return '↑'
    if x < 0:
        return '↓'
    return '→'
DIVIDER = '_' * 60
ABBREVIATIONS = [
    'Mr',
    'Mrs',
    'Ms',
    'Dr',
    'Prof',
    'Sr',
    'Jr',
    'St',
    'vs',
    'i.e',
    'e.g',
    'No',
    'Dept',
    'Inc',
    'Ltd',
    'Jan',
    'Feb',
    'Mar',
    'Apr',
    'Jun',
    'Jul',
    'Aug',
    'Sep',
    'Sept',
    'Oct',
    'Nov',
    'Dec',
    'Vol',
]
from shared.text import protect_abbreviations
from shared.text import restore_abbreviations
from prompting.analysis_utils import iter_sentence_spans_from_anno_text
from prompting.analysis_utils import build_subcategory_map


def pick_id_columns(df: pd.DataFrame) -> list:
    missing = [c for c in [COL_LETTER_ID, COL_SENTENCE_ID] if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required ID columns for merge: {missing}')
    return [COL_LETTER_ID, COL_SENTENCE_ID]


def build_level_df(path: str, context_label: str, level: str) -> tuple[pd.DataFrame, list]:
    df = read_file(path)
    missing = [c for c in [COL_LETTER_ID, COL_SENTENCE_ID, COL_GOLD, COL_PRED] if c not in df.columns]
    if missing:
        raise ValueError(f'File {os.path.basename(path)} missing columns: {missing}')
    df['context'] = context_label
    df['prompt_level'] = level
    id_cols = pick_id_columns(df)
    df['_gold_set'] = df[COL_GOLD].apply(parse_label_list)
    df['_pred_set'] = df[COL_PRED].apply(parse_label_list)
    if COL_PROBS_JSON in df.columns:
        df['_probs_dict'] = df[COL_PROBS_JSON].apply(parse_probs_dict)
    else:
        df['_probs_dict'] = [{} for _ in range(len(df))]
    keep_trace = [c for c in TRACE_COLS if c in df.columns]
    keep_cols = id_cols + keep_trace + ['_gold_set', '_pred_set', '_probs_dict']
    return (df[keep_cols].copy(), id_cols)


def merge_levels(df1: pd.DataFrame, df2: pd.DataFrame, id_cols: list) -> pd.DataFrame:
    merged = pd.merge(df1, df2, on=id_cols, how='inner', suffixes=('_a', '_b'))
    mismatches = int(merged['_gold_set_a'].ne(merged['_gold_set_b']).sum())
    if mismatches > 0:
        print(
            f' WARNING: {mismatches} instances have differing gold label sets between files (unexpected).',
        )
    merged['_gold_set'] = merged['_gold_set_a']
    return merged


def build_transition_delta_table(
    merged: pd.DataFrame,
    level_a: str,
    level_b: str,
    context_label: str,
) -> pd.DataFrame:
    gold_sets = merged['_gold_set']
    pred_a = merged['_pred_set_a']
    pred_b = merged['_pred_set_b']
    rows = []
    for cat in CATEGORIES:
        gold_has = gold_sets.apply(lambda s: cat in s)
        pred_a_has = pred_a.apply(lambda s: cat in s)
        pred_b_has = pred_b.apply(lambda s: cat in s)
        conf_a = compute_confusion(gold_has, pred_a_has)
        conf_b = compute_confusion(gold_has, pred_b_has)
        p_a = safe_precision(conf_a['TP'], conf_a['FP'])
        r_a = safe_recall(conf_a['TP'], conf_a['FN'])
        f1_a = f1_from_counts(conf_a['TP'], conf_a['FP'], conf_a['FN'])
        p_b = safe_precision(conf_b['TP'], conf_b['FP'])
        r_b = safe_recall(conf_b['TP'], conf_b['FN'])
        f1_b = f1_from_counts(conf_b['TP'], conf_b['FP'], conf_b['FN'])
        rows.append(
            {
                'transition': f'{level_a}→{level_b}',
                'transition_order': TRANSITION_ORDER[f'{level_a}→{level_b}'],
                'cat': cat,
                'category': CATEGORY_NAMES.get(cat, cat),
                'context': context_label,
                'context_order': CONTEXT_ORDER.get(context_label, 999),
                'ΔP': round(p_b - p_a, 4),
                'ΔR': round(r_b - r_a, 4),
                'ΔF1': round(f1_b - f1_a, 4),
            },
        )
    return pd.DataFrame(rows)


def print_console_transition_category_context(deltas_all: pd.DataFrame) -> None:
    deltas_all = deltas_all.sort_values(
        ['transition_order', 'cat', 'context_order'],
        ascending=[True, True, True],
    )
    for trans_label, sub_t in deltas_all.groupby('transition', sort=False):
        print('\n' + '=' * 92)
        print(f'TRANSITION {trans_label} | Per-category direction of change (ΔP / ΔR / ΔF1)')
        print('=' * 92)
        for cat, sub_c in sub_t.groupby('cat', sort=False):
            cat_name = sub_c['category'].iloc[0]
            print(f'\n{cat} {cat_name}')
            sub_c = sub_c.sort_values('context_order')
            for _, r in sub_c.iterrows():
                print(
                    f"  {r['context']:<10s}  ΔP {sign_arrow(r['ΔP'])} ({r['ΔP']:+.4f})   ΔR {sign_arrow(r['ΔR'])} ({r['ΔR']:+.4f})   ΔF1 {r['ΔF1']:+.4f}",
                )


def collect_instances_for_transition(
    merged: pd.DataFrame,
    level_a: str,
    level_b: str,
    context_label: str,
    subcat_map: dict,
) -> pd.DataFrame:
    id_cols = [COL_LETTER_ID, COL_SENTENCE_ID]
    transition_label = f'{level_a}→{level_b}'
    gold_sets = merged['_gold_set']
    pred_a = merged['_pred_set_a']
    pred_b = merged['_pred_set_b']
    trace_cols_a = [c for c in merged.columns if c.endswith('_a') and c.replace('_a', '') in TRACE_COLS]
    trace_cols_b = [c for c in merged.columns if c.endswith('_b') and c.replace('_b', '') in TRACE_COLS]
    rows = []
    for cat in CATEGORIES:
        cat_name = CATEGORY_NAMES.get(cat, cat)
        gold_has = gold_sets.apply(lambda s: cat in s)
        pred_a_has = pred_a.apply(lambda s: cat in s)
        pred_b_has = pred_b.apply(lambda s: cat in s)
        mask_fn_to_tp = gold_has & ~pred_a_has & pred_b_has
        mask_tp_to_fn = gold_has & pred_a_has & ~pred_b_has
        mask_tn_to_fp = ~gold_has & ~pred_a_has & pred_b_has
        mask_fp_to_tn = ~gold_has & pred_a_has & ~pred_b_has
        flow_specs = [
            ('FN→TP', mask_fn_to_tp),
            ('TP→FN', mask_tp_to_fn),
            ('TN→FP', mask_tn_to_fp),
            ('FP→TN', mask_fp_to_tn),
        ]
        for flow_name, mask in flow_specs:
            if int(mask.sum()) == 0:
                continue
            sub = merged.loc[mask, :].copy()
            sub['gold_set'] = sub['_gold_set'].apply(set_to_str)
            sub['pred_set_before'] = sub['_pred_set_a'].apply(set_to_str)
            sub['pred_set_after'] = sub['_pred_set_b'].apply(set_to_str)
            sub['gained_categories'] = sub.apply(
                lambda r: ','.join(
                    [
                        c2
                         for c2 in CATEGORIES if c2 in r['_pred_set_b'] and c2 not in r['_pred_set_a']
                    ],
                ),
                axis=1,
            )
            sub['lost_categories'] = sub.apply(
                lambda r: ','.join(
                    [
                        c2
                         for c2 in CATEGORIES if c2 in r['_pred_set_a'] and c2 not in r['_pred_set_b']
                    ],
                ),
                axis=1,
            )

            def _p(d, c2):
                try:
                    return float(d.get(c2, np.nan))
                except Exception:
                    return np.nan
            sub['prob_before'] = sub['_probs_dict_a'].apply(lambda d: _p(d, cat))
            sub['prob_after'] = sub['_probs_dict_b'].apply(lambda d: _p(d, cat))
            sub['delta_prob'] = sub['prob_after'] - sub['prob_before']
            sub['other_delta_probs_top'] = sub.apply(
                lambda r: delta_probs_summary(r['_probs_dict_a'], r['_probs_dict_b'], exclude_cat=cat),
                axis=1,
            )

            def lookup_subcats(row):
                lid = str(row[COL_LETTER_ID])
                sid = int(row[COL_SENTENCE_ID])
                return subcat_map.get((lid, sid, cat), [])
            sub['gold_subcategories'] = sub.apply(lookup_subcats, axis=1)
            sub['gold_subcategories_str'] = sub['gold_subcategories'].apply(
                lambda xs: '; '.join(xs) if isinstance(xs, list) and xs else '',
            )
            out = sub[id_cols + trace_cols_a + trace_cols_b + [
                'gold_set',
                'pred_set_before',
                'pred_set_after',
                'gained_categories',
                'lost_categories',
                'prob_before',
                'prob_after',
                'delta_prob',
                'other_delta_probs_top',
                'gold_subcategories_str',
            ]].copy()
            out.insert(0, 'transition', transition_label)
            out.insert(1, 'transition_order', TRANSITION_ORDER[transition_label])
            out.insert(2, 'gold_category', cat)
            out.insert(3, 'gold_category_name', cat_name)
            out.insert(4, 'flow', flow_name)
            out.insert(5, 'context', context_label)
            out.insert(6, 'context_order', CONTEXT_ORDER.get(context_label, 999))
            rows.append(out)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


def print_console_subcategory_flow_summary(df_instances_all: pd.DataFrame) -> None:
    required = {'transition_order', 'gold_category', 'context_order', 'flow', 'gold_subcategories_str'}
    missing = [c for c in required if c not in df_instances_all.columns]
    if missing:
        print(f'\nℹ Shift-only subcategory flow summary skipped (missing columns: {missing}).')
        return
    df = df_instances_all.copy()

    def split_subcats(s):
        s = str(s).strip()
        if not s:
            return [] if not SHOW_NONE_BUCKET else [SUBCAT_NONE]
        parts = [p.strip() for p in s.split(';') if p.strip()]
        return parts if parts else [SUBCAT_NONE] if SHOW_NONE_BUCKET else []
    df['__subcat_list'] = df['gold_subcategories_str'].apply(split_subcats)
    df = df.explode('__subcat_list')
    if df.empty:
        print('\nℹ No subcategory rows to report (all empty).')
        return
    counts = df.groupby(
        [
            'transition_order',
            'transition',
            'gold_category',
            'gold_category_name',
            'context_order',
            'context',
            'flow',
            '__subcat_list',
        ],
    ).size().reset_index(
        name='count',
    ).sort_values(
        ['transition_order', 'gold_category', 'context_order', 'flow', 'count'],
        ascending=[True, True, True, True, False],
    )
    print('\n' + '=' * 92)
    print('SHIFT-ONLY SUBCATEGORY DISTRIBUTIONS (absolute counts within shifted instances)')
    print('=' * 92)
    for (_, t_lab), sub_t in counts.groupby(['transition_order', 'transition'], sort=False):
        print('\n' + '-' * 92)
        print(f'TRANSITION {t_lab}')
        print('-' * 92)
        for cat, sub_c in sub_t.groupby('gold_category', sort=False):
            cat_name = sub_c['gold_category_name'].iloc[0]
            print(f'\n{cat} {cat_name}')
            for ctx, sub_ctx in sub_c.groupby('context', sort=False):
                print(f'  [{ctx}]')
                for flow, sub_f in sub_ctx.groupby('flow', sort=False):
                    top = sub_f.head(TOP_SUBCATS_ABS)[['__subcat_list', 'count']]
                    if top.empty:
                        continue
                    pairs = ', '.join(
                        [f"{r['__subcat_list']}={int(r['count'])}" for _, r in top.iterrows()],
                    )
                    print(f'    {flow:<5s}: {pairs}')


def split_subcats_or_bucket(subcats, gold_has: bool):
    if not gold_has:
        return [SUBCAT_GOLD_NEG]
    if not subcats:
        return [SUBCAT_NONE]
    return [s.strip() for s in subcats if isinstance(s, str) and s.strip()]


def build_subcategory_transition_stats(
    merged: pd.DataFrame,
    level_a: str,
    level_b: str,
    context_label: str,
    subcat_map: dict,
) -> pd.DataFrame:
    trans_label = f'{level_a}→{level_b}'
    gold_sets = merged['_gold_set']
    pred_a = merged['_pred_set_a']
    pred_b = merged['_pred_set_b']
    rows = []
    for cat in CATEGORIES:
        cat_name = CATEGORY_NAMES.get(cat, cat)
        gold_has = gold_sets.apply(lambda s: cat in s)
        pred_a_has = pred_a.apply(lambda s: cat in s)
        pred_b_has = pred_b.apply(lambda s: cat in s)
        fn_to_tp = gold_has & ~pred_a_has & pred_b_has
        tp_to_fn = gold_has & pred_a_has & ~pred_b_has
        tn_to_fp = ~gold_has & ~pred_a_has & pred_b_has
        fp_to_tn = ~gold_has & pred_a_has & ~pred_b_has
        tmp = merged[[COL_LETTER_ID, COL_SENTENCE_ID]].copy()
        tmp['gold_has'] = gold_has.values
        tmp['pred_a_has'] = pred_a_has.values
        tmp['pred_b_has'] = pred_b_has.values
        tmp['flow'] = 'unchanged'
        tmp.loc[fn_to_tp, 'flow'] = 'FN→TP'
        tmp.loc[tp_to_fn, 'flow'] = 'TP→FN'
        tmp.loc[tn_to_fp, 'flow'] = 'TN→FP'
        tmp.loc[fp_to_tn, 'flow'] = 'FP→TN'

        def _subcats(row):
            key = (str(row[COL_LETTER_ID]), int(row[COL_SENTENCE_ID]), cat)
            return subcat_map.get(key, [])
        tmp['subcat_list'] = tmp.apply(_subcats, axis=1)
        tmp['subcategory'] = tmp.apply(lambda r: split_subcats_or_bucket(r['subcat_list'], bool(r['gold_has'])), axis=1)
        tmp = tmp.explode('subcategory').reset_index(drop=True)
        gold_pos = tmp[tmp['gold_has']].copy()
        support = gold_pos.groupby('subcategory').size().reset_index(name='support_gold')
        tp_counts = gold_pos.groupby('subcategory').agg(
            tp_before=('pred_a_has', 'sum'),
            tp_after=('pred_b_has', 'sum'),
        ).reset_index()
        tp_counts['tp_before'] = tp_counts['tp_before'].astype(int)
        tp_counts['tp_after'] = tp_counts['tp_after'].astype(int)
        rec = tp_counts.merge(support, on='subcategory', how='left')
        rec['recall_before'] = np.where(rec['support_gold'] > 0, rec['tp_before'] / rec['support_gold'], np.nan)
        rec['recall_after'] = np.where(rec['support_gold'] > 0, rec['tp_after'] / rec['support_gold'], np.nan)
        rec['delta_recall'] = rec['recall_after'] - rec['recall_before']
        rec['delta_tp'] = rec['tp_after'] - rec['tp_before']
        flow_counts = tmp.groupby(['subcategory', 'flow']).size().unstack(fill_value=0).reset_index()
        out = flow_counts.merge(rec, on='subcategory', how='left')
        for flow in ['FN→TP', 'TP→FN', 'TN→FP', 'FP→TN', 'unchanged']:
            if flow not in out.columns:
                out[flow] = 0
        out = out.rename(
            columns={
                'FN→TP': 'count_FN→TP',
                'TP→FN': 'count_TP→FN',
                'TN→FP': 'count_TN→FP',
                'FP→TN': 'count_FP→TN',
                'unchanged': 'count_unchanged',
            },
        )
        out['support_gold'] = out['support_gold'].fillna(0).astype(int)
        out['tp_before'] = out['tp_before'].fillna(0).astype(int)
        out['tp_after'] = out['tp_after'].fillna(0).astype(int)
        out['delta_tp'] = out['delta_tp'].fillna(0).astype(int)
        for c0 in ['recall_before', 'recall_after', 'delta_recall']:
            if c0 in out.columns:
                out[c0] = out[c0].astype(float)
        out['rate_FN→TP'] = np.where(out['support_gold'] > 0, out['count_FN→TP'] / out['support_gold'], np.nan)
        out['rate_TP→FN'] = np.where(out['support_gold'] > 0, out['count_TP→FN'] / out['support_gold'], np.nan)
        out.insert(0, 'transition', trans_label)
        out.insert(1, 'transition_order', TRANSITION_ORDER[trans_label])
        out.insert(2, 'category_code', cat)
        out.insert(3, 'category_name', cat_name)
        out.insert(4, 'context', context_label)
        out.insert(5, 'context_order', CONTEXT_ORDER.get(context_label, 999))
        rows.append(out)
    if not rows:
        return pd.DataFrame()
    out_all = pd.concat(rows, ignore_index=True)
    out_all = out_all.sort_values(
        ['transition_order', 'category_code', 'context_order', 'support_gold', 'subcategory'],
        ascending=[True, True, True, False, True],
    )
    return out_all


def print_console_subcat_overview(subcat_stats: pd.DataFrame) -> None:
    if subcat_stats is None or subcat_stats.empty:
        print('\nℹ No subcategory stats to print.')
        return
    subcat_stats = subcat_stats.sort_values(
        ['transition_order', 'category_code', 'context_order'],
        ascending=[True, True, True],
    )
    print('\n' + '=' * 92)
    print('SUBCATEGORY OVERVIEW (support, recall counts, and normalised flow rates)')
    print('=' * 92)
    for trans, sub_t in subcat_stats.groupby('transition', sort=False):
        print('\n' + '-' * 92)
        print(f'TRANSITION {trans}')
        print('-' * 92)
        for cat, sub_c in sub_t.groupby('category_code', sort=False):
            cat_name = sub_c['category_name'].iloc[0]
            print(f'\n{cat} {cat_name}')
            for ctx, sub_ctx in sub_c.groupby('context', sort=False):
                print(f'  [{ctx}]')
                goldpos = sub_ctx[sub_ctx['support_gold'] > 0].copy()

                def _line(r, which: str):
                    if which == 'FN→TP':
                        cnt = int(r['count_FN→TP'])
                        rate = r['rate_FN→TP']
                    else:
                        cnt = int(r['count_TP→FN'])
                        rate = r['rate_TP→FN']
                    return f"      {r['subcategory']}: {r['tp_before']}/{r['support_gold']} ({r['recall_before']:.3f}) -> {r['tp_after']}/{r['support_gold']} ({r['recall_after']:.3f}) | ΔTP {int(r['delta_tp']):+d} | ΔR {r['delta_recall']:+.3f} | {which} {cnt}/{r['support_gold']} ({rate:.3f})"
                top_fn_tp = goldpos.sort_values('rate_FN→TP', ascending=False).head(TOP_SUBCATS_OVERVIEW)
                if not top_fn_tp.empty:
                    print('    Top FN→TP (normalised by gold support):')
                    for _, r in top_fn_tp.iterrows():
                        print(_line(r, 'FN→TP'))
                top_tp_fn = goldpos.sort_values('rate_TP→FN', ascending=False).head(TOP_SUBCATS_OVERVIEW)
                if not top_tp_fn.empty:
                    print('    Top TP→FN (normalised by gold support):')
                    for _, r in top_tp_fn.iterrows():
                        print(_line(r, 'TP→FN'))
                neg_bucket = sub_ctx[sub_ctx['subcategory'] == SUBCAT_GOLD_NEG]
                if not neg_bucket.empty:
                    r0 = neg_bucket.iloc[0]
                    fp_tn = int(r0.get('count_FP→TN', 0))
                    tn_fp = int(r0.get('count_TN→FP', 0))
                    if fp_tn or tn_fp:
                        print(
                            f'    {SUBCAT_GOLD_NEG}: TN→FP={tn_fp}, FP→TN={fp_tn} (not normalised)',
                        )


def main():
    os.makedirs(EXPORT_DIR, exist_ok=True)
    print(' Building subcategory map from letters pickle (this may take a moment)...')
    subcat_map = build_subcategory_map()
    print(f' Subcategory map built: {len(subcat_map):,} (letter_id, sentence_id, category) keys')
    instances_by_transition = {lab: [] for lab in TRANSITION_LABELS}
    deltas_all = []
    subcat_stats_all = []
    for context_label, context_dir in CONTEXT_DIRS.items():
        if not os.path.isdir(context_dir):
            print(f' Skipping missing directory: {context_dir}')
            continue
        level_dfs = {}
        id_cols_ref = None
        for lvl in LEVELS:
            path = os.path.join(context_dir, CSV_PATTERN.format(level=lvl))
            if not os.path.isfile(path):
                raise FileNotFoundError(f'Missing expected file: {path}')
            df_lvl, id_cols = build_level_df(path, context_label=context_label, level=lvl)
            level_dfs[lvl] = df_lvl
            if id_cols_ref is None:
                id_cols_ref = id_cols
        for a, b in TRANSITIONS:
            trans_label = f'{a}→{b}'
            merged = merge_levels(level_dfs[a], level_dfs[b], id_cols_ref)
            if merged.empty:
                print(
                    f' Empty merge for {context_label} {trans_label}. Check (letter_id, sentence_id) alignment.',
                )
                continue
            deltas_all.append(build_transition_delta_table(merged, a, b, context_label))
            subcat_stats_all.append(
                build_subcategory_transition_stats(
                    merged=merged,
                    level_a=a,
                    level_b=b,
                    context_label=context_label,
                    subcat_map=subcat_map,
                ),
            )
            df_instances = collect_instances_for_transition(
                merged,
                a,
                b,
                context_label=context_label,
                subcat_map=subcat_map,
            )
            if not df_instances.empty:
                instances_by_transition[trans_label].append(df_instances)
    if deltas_all:
        deltas_all_df = pd.concat(deltas_all, ignore_index=True)
        print_console_transition_category_context(deltas_all_df)
    else:
        print('\nNo delta summaries produced (no merges succeeded).')
    combined_all = []
    for trans_label in TRANSITION_LABELS:
        parts = instances_by_transition.get(trans_label, [])
        if not parts:
            continue
        df_t = pd.concat(parts, ignore_index=True)
        df_t = df_t.sort_values(
            [
                'transition_order',
                'gold_category',
                'flow',
                COL_LETTER_ID,
                COL_SENTENCE_ID,
                'context_order',
            ],
            ascending=[True, True, True, True, True, True],
        )
        out_name = OUT_TRANSITION_PATTERN.format(transition=trans_label.replace('→', '_to_'))
        out_path = os.path.join(EXPORT_DIR, out_name)
        df_t.to_csv(out_path, index=False)
        print(f'\n Saved transition CSV (both contexts): {out_path}  (rows={len(df_t):,})')
        combined_all.append(df_t)
    if combined_all:
        df_all = pd.concat(combined_all, ignore_index=True)
        df_all = df_all.sort_values(
            [
                'transition_order',
                'gold_category',
                'flow',
                COL_LETTER_ID,
                COL_SENTENCE_ID,
                'context_order',
            ],
            ascending=[True, True, True, True, True, True],
        )
        df_all.to_csv(OUT_COMBINED, index=False)
        print(f'\n Saved combined instances CSV: {OUT_COMBINED}  (rows={len(df_all):,})')
        print_console_subcategory_flow_summary(df_all)
    else:
        print('\nNo shifting instances found (nothing to export).')
    if subcat_stats_all:
        subcat_stats_df = pd.concat(subcat_stats_all, ignore_index=True)
        subcat_stats_df = subcat_stats_df.sort_values(
            ['transition_order', 'category_code', 'context_order', 'support_gold', 'subcategory'],
            ascending=[True, True, True, False, True],
        )
        print_console_subcat_overview(subcat_stats_df)
        out_subcat_csv = os.path.join(EXPORT_DIR, 'transition_subcategory_stats_all.csv')
        subcat_stats_df.to_csv(out_subcat_csv, index=False)
        print(f'\n Saved subcategory stats CSV: {out_subcat_csv}  (rows={len(subcat_stats_df):,})')
    else:
        print('\nℹ No subcategory stats computed (no merges succeeded).')
if __name__ == '__main__':
    main()


## Subcategory changes

In [ ]:
from __future__ import annotations
import os
import re
import ast
import math
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
from shared.utils import load_pickle

BASE_DIR = str(paths.PROMPT_RESULTS / 'sentence')
CONTEXT_DIRS = {'nocontext': os.path.join(BASE_DIR, 'nocontext'), 'context_4': os.path.join(BASE_DIR, 'context_4')}
CONTEXT_ORDER = {'nocontext': 0, 'context_4': 1}
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
LEVEL_ORDER = {lvl: i for i, lvl in enumerate(LEVELS)}
CSV_PATTERN = 'llm_results_sentence_multilabel_{level}.csv'
COL_LETTER_ID = 'letter_id'
COL_SENTENCE_ID = 'sentence_id'
COL_GOLD = 'gold_categories'
COL_PRED = 'predicted_tokens'
BERT_SENTENCE_CSV = str(paths.sentence_run(4) / 'general/all_folds_predictions.csv')
BERT_COL_LETTER_ID = 'letter_id'
BERT_COL_SENTENCE_ID = 'sentence_id'
BERT_COL_TRUE = 'true_labels'
BERT_COL_PRED = 'pred_labels'
BERT_INDEX_TO_CODE = {0: 'D', 1: 'C', 2: 'F', 3: 'G', 4: 'I', 5: 'A', 6: 'H', 7: 'E', 8: 'B'}
CATEGORIES = list('ABCDEFGHI')
CATEGORY_NAMES = {
    'A': 'Inspirations and Influences',
    'B': 'Writing Process and Creative Insight',
    'C': 'Elements of Fictional Craft',
    'D': 'Editorial and Publishing Processes',
    'E': 'Quality and Evaluation',
    'F': 'Financial and Legal Aspects',
    'G': 'Gratitude and Acknowledgement',
    'H': 'Marketing and Promotion',
    'I': 'Impact and Reception',
    'J': 'No reference to Dickens’s works present',
}
CATEGORY_CODE_BY_NAME = {v: k for k, v in CATEGORY_NAMES.items() if k in CATEGORIES}
EXPORT_DIR = os.path.join(BASE_DIR, 'subcategory_recognition')
OUT_STATS_CSV = os.path.join(EXPORT_DIR, 'subcategory_recognition_stats_all.csv')
OUT_RECOVERED_INSTANCES_CSV = os.path.join(EXPORT_DIR, 'bert_missed_llama_recovered_instances.csv')
SHOW_NONE_BUCKET = True
SUBCAT_NONE = '(none)'
PRINT_ALL_SUBCATS = True
TOP_SUBCATS_PER_CATEGORY = 15
ADD_CATEGORY_TOTAL_ROWS = True
CATEGORY_TOTAL_LABEL = '(all subcategories)'
SORT_SUBCATS_BY = 'llm_support_gold'
from prompting.analysis_utils import read_file
from prompting.analysis_utils import parse_known_categories as parse_label_list


def parse_bert_multihot_to_codes(x) -> set:
    if pd.isna(x):
        return set()
    if isinstance(x, (list, tuple, np.ndarray)):
        vals = list(x)
    else:
        s = str(x).strip()
        if not s:
            return set()
        try:
            vals = ast.literal_eval(s)
        except Exception:
            return set()
    if not isinstance(vals, (list, tuple)) or len(vals) != 9:
        return set()
    out = set()
    for idx, v in enumerate(vals):
        try:
            fv = float(v)
        except Exception:
            continue
        if fv >= 0.5:
            code = BERT_INDEX_TO_CODE.get(idx)
            if code in CATEGORIES:
                out.add(code)
    return out
from prompting.analysis_utils import set_to_str


def is_na(x) -> bool:
    try:
        return pd.isna(x)
    except Exception:
        return x is None
DIVIDER = '_' * 60
ABBREVIATIONS = [
    'Mr',
    'Mrs',
    'Ms',
    'Dr',
    'Prof',
    'Sr',
    'Jr',
    'St',
    'vs',
    'i.e',
    'e.g',
    'No',
    'Dept',
    'Inc',
    'Ltd',
    'Jan',
    'Feb',
    'Mar',
    'Apr',
    'Jun',
    'Jul',
    'Aug',
    'Sep',
    'Sept',
    'Oct',
    'Nov',
    'Dec',
    'Vol',
]
from shared.text import protect_abbreviations
from shared.text import restore_abbreviations
from prompting.analysis_utils import iter_sentence_spans_from_anno_text
from prompting.analysis_utils import build_subcategory_map


def build_llm_level_df(path: str, context_label: str, level: str) -> pd.DataFrame:
    df = read_file(path)
    missing = [c for c in [COL_LETTER_ID, COL_SENTENCE_ID, COL_GOLD, COL_PRED] if c not in df.columns]
    if missing:
        raise ValueError(f'File {os.path.basename(path)} missing columns: {missing}')
    df['context'] = context_label
    df['prompt_level'] = level
    df[COL_LETTER_ID] = df[COL_LETTER_ID].astype(str)
    df[COL_SENTENCE_ID] = df[COL_SENTENCE_ID].astype(float).astype(int)
    df['_gold_set'] = df[COL_GOLD].apply(parse_label_list)
    df['_pred_set'] = df[COL_PRED].apply(parse_label_list)
    return df[[COL_LETTER_ID, COL_SENTENCE_ID, 'context', 'prompt_level', '_gold_set', '_pred_set']].copy()


def load_bert_sentence_df(path: str) -> pd.DataFrame:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'BERT sentence-level CSV not found: {path}')
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    required = {BERT_COL_LETTER_ID, BERT_COL_SENTENCE_ID, BERT_COL_TRUE, BERT_COL_PRED}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(
            f'BERT CSV missing required columns: {sorted(missing)}\nAvailable columns: {list(df.columns)}',
        )
    out = df[[BERT_COL_LETTER_ID, BERT_COL_SENTENCE_ID, BERT_COL_TRUE, BERT_COL_PRED]].copy()
    out[BERT_COL_LETTER_ID] = out[BERT_COL_LETTER_ID].astype(str)
    out[BERT_COL_SENTENCE_ID] = out[BERT_COL_SENTENCE_ID].astype(float).astype(int)
    out['_bert_gold_set'] = out[BERT_COL_TRUE].apply(parse_bert_multihot_to_codes)
    out['_bert_pred_set'] = out[BERT_COL_PRED].apply(parse_bert_multihot_to_codes)
    out = out.rename(columns={BERT_COL_LETTER_ID: COL_LETTER_ID, BERT_COL_SENTENCE_ID: COL_SENTENCE_ID})
    out = out[[COL_LETTER_ID, COL_SENTENCE_ID, '_bert_gold_set', '_bert_pred_set']].copy()
    dup_mask = out.duplicated(subset=[COL_LETTER_ID, COL_SENTENCE_ID], keep=False)
    n_dups = int(dup_mask.sum())
    if n_dups > 0:
        print(
            f'\nWARNING: BERT CSV contains {n_dups} duplicate rows on ({COL_LETTER_ID}, {COL_SENTENCE_ID}).',
        )
        print('Keeping the last occurrence for each duplicate key.')
        out = out.drop_duplicates(subset=[COL_LETTER_ID, COL_SENTENCE_ID], keep='last').copy()
    return out


def split_subcats_gold_positive(subcats):
    if not subcats:
        return [SUBCAT_NONE] if SHOW_NONE_BUCKET else []
    parts = [s.strip() for s in subcats if isinstance(s, str) and s.strip()]
    if parts:
        return parts
    return [SUBCAT_NONE] if SHOW_NONE_BUCKET else []


def attach_and_explode_subcategories(df: pd.DataFrame, cat: str, subcat_map: dict) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    tmp = df.copy()

    def _subcats(row):
        key = (str(row[COL_LETTER_ID]), int(row[COL_SENTENCE_ID]), cat)
        return subcat_map.get(key, [])
    tmp['subcat_list'] = tmp.apply(_subcats, axis=1)
    tmp['subcategory'] = tmp['subcat_list'].apply(split_subcats_gold_positive)
    tmp = tmp.explode('subcategory').reset_index(drop=True)
    if tmp.empty:
        return tmp
    tmp['all_gold_subcategories_str'] = tmp['subcat_list'].apply(lambda xs: '; '.join(xs) if isinstance(xs, list) and xs else '')
    return tmp


def aggregate_support_success_by_subcategory(
    df_gold_positive: pd.DataFrame,
    cat: str,
    subcat_map: dict,
    success_col: str,
    support_col_name: str,
    success_col_name: str,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    total_support = int(len(df_gold_positive))
    total_success = int(df_gold_positive[success_col].sum()) if total_support > 0 else 0
    total_fail = total_support - total_success
    total_rate = total_success / total_support if total_support > 0 else np.nan
    total_dict = {
        'subcategory': CATEGORY_TOTAL_LABEL,
        support_col_name: total_support,
        success_col_name: total_success,
        'missed_or_failed': total_fail,
        'rate': total_rate,
    }
    if df_gold_positive.empty:
        return (pd.DataFrame(columns=['subcategory', support_col_name, success_col_name]), total_dict)
    exploded = attach_and_explode_subcategories(df_gold_positive, cat=cat, subcat_map=subcat_map)
    if exploded.empty:
        return (pd.DataFrame(columns=['subcategory', support_col_name, success_col_name]), total_dict)
    agg = exploded.groupby('subcategory').agg(
        _support=(success_col, 'size'),
        _success=(success_col, 'sum'),
    ).reset_index()
    agg['_support'] = agg['_support'].astype(int)
    agg['_success'] = agg['_success'].astype(int)
    agg[support_col_name] = agg['_support']
    agg[success_col_name] = agg['_success']
    agg = agg.drop(columns=['_support', '_success'])
    return (agg, total_dict)


def build_subcategory_stats_for_level(
    df_llm_level: pd.DataFrame,
    context_label: str,
    level: str,
    subcat_map: dict,
    df_bert: Optional[pd.DataFrame]=None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows_out = []
    recovered_instances_out = []
    shared = None
    if df_bert is not None and (not df_bert.empty):
        shared = pd.merge(
            df_llm_level[[COL_LETTER_ID, COL_SENTENCE_ID, '_gold_set', '_pred_set']],
            df_bert[[COL_LETTER_ID, COL_SENTENCE_ID, '_bert_gold_set', '_bert_pred_set']],
            on=[COL_LETTER_ID, COL_SENTENCE_ID],
            how='inner',
            validate='one_to_one',
        )
    for cat in CATEGORIES:
        cat_name = CATEGORY_NAMES.get(cat, cat)
        llm_tmp = df_llm_level[[COL_LETTER_ID, COL_SENTENCE_ID, '_gold_set', '_pred_set']].copy()
        llm_tmp['gold_has'] = llm_tmp['_gold_set'].apply(lambda s: cat in s)
        llm_tmp['pred_has'] = llm_tmp['_pred_set'].apply(lambda s: cat in s)
        llm_gold_pos = llm_tmp[llm_tmp['gold_has']].copy()
        llm_agg, llm_total = aggregate_support_success_by_subcategory(
            df_gold_positive=llm_gold_pos,
            cat=cat,
            subcat_map=subcat_map,
            success_col='pred_has',
            support_col_name='llm_support_gold',
            success_col_name='llm_recognized_tp',
        )
        if not llm_agg.empty:
            llm_agg['llm_missed_fn'] = llm_agg['llm_support_gold'] - llm_agg['llm_recognized_tp']
            llm_agg['llm_recall'] = np.where(
                llm_agg['llm_support_gold'] > 0,
                llm_agg['llm_recognized_tp'] / llm_agg['llm_support_gold'],
                np.nan,
            )
        else:
            llm_agg = pd.DataFrame(
                columns=[
                    'subcategory',
                    'llm_support_gold',
                    'llm_recognized_tp',
                    'llm_missed_fn',
                    'llm_recall',
                ],
            )
        llm_total_row = {
            'subcategory': CATEGORY_TOTAL_LABEL,
            'llm_support_gold': llm_total['llm_support_gold'] if 'llm_support_gold' in llm_total else llm_total.get('llm_support_gold', None),
            'llm_recognized_tp': llm_total['llm_recognized_tp'] if 'llm_recognized_tp' in llm_total else llm_total.get('llm_recognized_tp', None),
            'llm_missed_fn': llm_total['missed_or_failed'],
            'llm_recall': llm_total['rate'],
        }
        llm_total_row['llm_support_gold'] = llm_total.get(
            'llm_support_gold',
            llm_total.get('llm_support_gold', llm_total.get('llm_support_gold')),
        )
        llm_total_row['llm_recognized_tp'] = llm_total.get(
            'llm_recognized_tp',
            llm_total.get('llm_recognized_tp', llm_total.get('llm_recognized_tp')),
        )
        if llm_total_row['llm_support_gold'] is None:
            llm_total_row['llm_support_gold'] = int(len(llm_gold_pos))
        if llm_total_row['llm_recognized_tp'] is None:
            llm_total_row['llm_recognized_tp'] = int(llm_gold_pos['pred_has'].sum()) if not llm_gold_pos.empty else 0
        if df_bert is not None and (not df_bert.empty):
            bert_tmp = df_bert[[COL_LETTER_ID, COL_SENTENCE_ID, '_bert_gold_set', '_bert_pred_set']].copy()
            bert_tmp['bert_gold_has'] = bert_tmp['_bert_gold_set'].apply(lambda s: cat in s)
            bert_tmp['bert_pred_has'] = bert_tmp['_bert_pred_set'].apply(lambda s: cat in s)
            bert_gold_pos = bert_tmp[bert_tmp['bert_gold_has']].copy()
            bert_agg, bert_total = aggregate_support_success_by_subcategory(
                df_gold_positive=bert_gold_pos,
                cat=cat,
                subcat_map=subcat_map,
                success_col='bert_pred_has',
                support_col_name='bert_support_gold',
                success_col_name='bert_recognized_tp',
            )
            if not bert_agg.empty:
                bert_agg['bert_missed_fn'] = bert_agg['bert_support_gold'] - bert_agg['bert_recognized_tp']
                bert_agg['bert_recall'] = np.where(
                    bert_agg['bert_support_gold'] > 0,
                    bert_agg['bert_recognized_tp'] / bert_agg['bert_support_gold'],
                    np.nan,
                )
            else:
                bert_agg = pd.DataFrame(
                    columns=[
                        'subcategory',
                        'bert_support_gold',
                        'bert_recognized_tp',
                        'bert_missed_fn',
                        'bert_recall',
                    ],
                )
            bert_total_row = {
                'subcategory': CATEGORY_TOTAL_LABEL,
                'bert_support_gold': bert_total.get('bert_support_gold', int(len(bert_gold_pos))),
                'bert_recognized_tp': bert_total.get(
                    'bert_recognized_tp',
                    int(bert_gold_pos['bert_pred_has'].sum()) if not bert_gold_pos.empty else 0,
                ),
                'bert_missed_fn': bert_total['missed_or_failed'],
                'bert_recall': bert_total['rate'],
            }
        else:
            bert_agg = pd.DataFrame(
                columns=[
                    'subcategory',
                    'bert_support_gold',
                    'bert_recognized_tp',
                    'bert_missed_fn',
                    'bert_recall',
                ],
            )
            bert_total_row = {
                'subcategory': CATEGORY_TOTAL_LABEL,
                'bert_support_gold': np.nan,
                'bert_recognized_tp': np.nan,
                'bert_missed_fn': np.nan,
                'bert_recall': np.nan,
            }
        if shared is not None and (not shared.empty):
            rec_tmp = shared[[
                COL_LETTER_ID,
                COL_SENTENCE_ID,
                '_gold_set',
                '_pred_set',
                '_bert_gold_set',
                '_bert_pred_set',
            ]].copy()
            rec_tmp['llm_gold_has'] = rec_tmp['_gold_set'].apply(lambda s: cat in s)
            rec_tmp['llm_pred_has'] = rec_tmp['_pred_set'].apply(lambda s: cat in s)
            rec_tmp['bert_gold_has'] = rec_tmp['_bert_gold_set'].apply(lambda s: cat in s)
            rec_tmp['bert_pred_has'] = rec_tmp['_bert_pred_set'].apply(lambda s: cat in s)
            bert_missed_shared = rec_tmp[rec_tmp['bert_gold_has'] & ~rec_tmp['bert_pred_has']].copy()
            rec_agg, rec_total = aggregate_support_success_by_subcategory(
                df_gold_positive=bert_missed_shared,
                cat=cat,
                subcat_map=subcat_map,
                success_col='llm_pred_has',
                support_col_name='bert_missed_support_shared',
                success_col_name='llama_recovered_from_bert_missed',
            )
            if not rec_agg.empty:
                rec_agg['bert_missed_not_recovered_shared'] = rec_agg['bert_missed_support_shared'] - rec_agg['llama_recovered_from_bert_missed']
                rec_agg['recovery_rate'] = np.where(
                    rec_agg['bert_missed_support_shared'] > 0,
                    rec_agg['llama_recovered_from_bert_missed'] / rec_agg['bert_missed_support_shared'],
                    np.nan,
                )
            else:
                rec_agg = pd.DataFrame(
                    columns=[
                        'subcategory',
                        'bert_missed_support_shared',
                        'llama_recovered_from_bert_missed',
                        'bert_missed_not_recovered_shared',
                        'recovery_rate',
                    ],
                )
            rec_total_row = {
                'subcategory': CATEGORY_TOTAL_LABEL,
                'bert_missed_support_shared': rec_total.get('bert_missed_support_shared', int(len(bert_missed_shared))),
                'llama_recovered_from_bert_missed': rec_total.get(
                    'llama_recovered_from_bert_missed',
                    int(bert_missed_shared['llm_pred_has'].sum()) if not bert_missed_shared.empty else 0,
                ),
                'bert_missed_not_recovered_shared': rec_total['missed_or_failed'],
                'recovery_rate': rec_total['rate'],
            }
            recovered = bert_missed_shared[bert_missed_shared['llm_pred_has']].copy()
            if not recovered.empty:
                rec_exp = attach_and_explode_subcategories(recovered, cat=cat, subcat_map=subcat_map)
                if not rec_exp.empty:
                    rec_exp['context'] = context_label
                    rec_exp['context_order'] = CONTEXT_ORDER.get(context_label, 999)
                    rec_exp['prompt_level'] = level
                    rec_exp['prompt_order'] = LEVEL_ORDER.get(level, 999)
                    rec_exp['category_code'] = cat
                    rec_exp['category_name'] = cat_name
                    rec_exp['llm_gold_set_str'] = rec_exp['_gold_set'].apply(set_to_str)
                    rec_exp['llm_pred_set_str'] = rec_exp['_pred_set'].apply(set_to_str)
                    rec_exp['bert_gold_set_str'] = rec_exp['_bert_gold_set'].apply(set_to_str)
                    rec_exp['bert_pred_set_str'] = rec_exp['_bert_pred_set'].apply(set_to_str)
                    rec_exp['llm_gold_has_category'] = rec_exp['llm_gold_has'].astype(bool)
                    rec_exp['llm_pred_has_category'] = rec_exp['llm_pred_has'].astype(bool)
                    rec_exp['bert_gold_has_category'] = rec_exp['bert_gold_has'].astype(bool)
                    rec_exp['bert_pred_has_category'] = rec_exp['bert_pred_has'].astype(bool)
                    recovered_instances_out.append(
                        rec_exp[[
                            'context',
                            'context_order',
                            'prompt_level',
                            'prompt_order',
                            'category_code',
                            'category_name',
                            COL_LETTER_ID,
                            COL_SENTENCE_ID,
                            'subcategory',
                            'all_gold_subcategories_str',
                            'llm_gold_set_str',
                            'llm_pred_set_str',
                            'bert_gold_set_str',
                            'bert_pred_set_str',
                            'llm_gold_has_category',
                            'llm_pred_has_category',
                            'bert_gold_has_category',
                            'bert_pred_has_category',
                        ]].copy(),
                    )
        else:
            rec_agg = pd.DataFrame(
                columns=[
                    'subcategory',
                    'bert_missed_support_shared',
                    'llama_recovered_from_bert_missed',
                    'bert_missed_not_recovered_shared',
                    'recovery_rate',
                ],
            )
            rec_total_row = {
                'subcategory': CATEGORY_TOTAL_LABEL,
                'bert_missed_support_shared': np.nan,
                'llama_recovered_from_bert_missed': np.nan,
                'bert_missed_not_recovered_shared': np.nan,
                'recovery_rate': np.nan,
            }
        merged = pd.DataFrame({'subcategory': []})
        for part in [llm_agg, bert_agg, rec_agg]:
            if part is not None and (not part.empty):
                if merged.empty:
                    merged = part.copy()
                else:
                    merged = merged.merge(part, on='subcategory', how='outer')
        if merged.empty:
            merged = pd.DataFrame(
                columns=[
                    'subcategory',
                    'llm_support_gold',
                    'llm_recognized_tp',
                    'llm_missed_fn',
                    'llm_recall',
                    'bert_support_gold',
                    'bert_recognized_tp',
                    'bert_missed_fn',
                    'bert_recall',
                    'bert_missed_support_shared',
                    'llama_recovered_from_bert_missed',
                    'bert_missed_not_recovered_shared',
                    'recovery_rate',
                ],
            )
        if ADD_CATEGORY_TOTAL_ROWS:
            total_row = pd.DataFrame(
                [
                    {
                        'subcategory': CATEGORY_TOTAL_LABEL,
                        'llm_support_gold': llm_total_row.get('llm_support_gold', np.nan),
                        'llm_recognized_tp': llm_total_row.get('llm_recognized_tp', np.nan),
                        'llm_missed_fn': llm_total_row.get('llm_missed_fn', np.nan),
                        'llm_recall': llm_total_row.get('llm_recall', np.nan),
                        'bert_support_gold': bert_total_row.get('bert_support_gold', np.nan),
                        'bert_recognized_tp': bert_total_row.get('bert_recognized_tp', np.nan),
                        'bert_missed_fn': bert_total_row.get('bert_missed_fn', np.nan),
                        'bert_recall': bert_total_row.get('bert_recall', np.nan),
                        'bert_missed_support_shared': rec_total_row.get('bert_missed_support_shared', np.nan),
                        'llama_recovered_from_bert_missed': rec_total_row.get('llama_recovered_from_bert_missed', np.nan),
                        'bert_missed_not_recovered_shared': rec_total_row.get('bert_missed_not_recovered_shared', np.nan),
                        'recovery_rate': rec_total_row.get('recovery_rate', np.nan),
                    },
                ],
            )
            merged = pd.concat(
                [total_row, merged[merged['subcategory'] != CATEGORY_TOTAL_LABEL]],
                ignore_index=True,
            )
        merged.insert(0, 'context', context_label)
        merged.insert(1, 'context_order', CONTEXT_ORDER.get(context_label, 999))
        merged.insert(2, 'prompt_level', level)
        merged.insert(3, 'prompt_order', LEVEL_ORDER.get(level, 999))
        merged.insert(4, 'category_code', cat)
        merged.insert(5, 'category_name', cat_name)
        merged['is_total_row'] = (merged['subcategory'] == CATEGORY_TOTAL_LABEL).astype(int)
        if SORT_SUBCATS_BY == 'bert_support_gold':
            sort_metric = 'bert_support_gold'
        elif SORT_SUBCATS_BY == 'recovery_support':
            sort_metric = 'bert_missed_support_shared'
        else:
            sort_metric = 'llm_support_gold'
        total_part = merged[merged['is_total_row'] == 1].copy()
        sub_part = merged[merged['is_total_row'] == 0].copy()
        if not sub_part.empty:
            sub_part[sort_metric] = pd.to_numeric(sub_part[sort_metric], errors='coerce')
            sub_part['llm_support_gold'] = pd.to_numeric(sub_part['llm_support_gold'], errors='coerce')
            sub_part['bert_support_gold'] = pd.to_numeric(sub_part['bert_support_gold'], errors='coerce')
            sub_part = sub_part.sort_values(
                [sort_metric, 'llm_support_gold', 'bert_support_gold', 'subcategory'],
                ascending=[False, False, False, True],
                na_position='last',
            )
        merged = pd.concat([total_part, sub_part], ignore_index=True)
        rows_out.append(merged)
    stats_df = pd.concat(rows_out, ignore_index=True) if rows_out else pd.DataFrame()
    recovered_df = pd.concat(recovered_instances_out, ignore_index=True) if recovered_instances_out else pd.DataFrame()
    return (stats_df, recovered_df)


def _fmt_side(prefix: str, tp, support, recall, missed) -> str:
    if is_na(support) or is_na(tp):
        return f'{prefix} n/a'
    support_i = int(support)
    tp_i = int(tp)
    missed_i = int(missed) if not is_na(missed) else max(support_i - tp_i, 0)
    if is_na(recall):
        recall_str = 'n/a'
    else:
        recall_str = f'{float(recall):.3f}'
    return f'{prefix} {tp_i}/{support_i} recognised (recall={recall_str}), missed={missed_i}'


def _fmt_recovery(prefix: str, recovered, support, rate) -> str:
    if is_na(support) or is_na(recovered):
        return f'{prefix} n/a'
    support_i = int(support)
    rec_i = int(recovered)
    rate_str = 'n/a' if is_na(rate) else f'{float(rate):.3f}'
    return f'{prefix} {rec_i}/{support_i} (rate={rate_str})'


def print_console_summary(stats_df: pd.DataFrame) -> None:
    if stats_df is None or stats_df.empty:
        print('\nNo subcategory recognition stats to print.')
        return
    stats_df = stats_df.sort_values(
        ['context_order', 'prompt_order', 'category_code', 'is_total_row', 'subcategory'],
        ascending=[True, True, True, False, True],
    )
    print('\n' + '=' * 120)
    print(
        'SUBCATEGORY RECOGNITION COUNTS (no transition logic) + BERT baseline + BERT-missed -> Llama recovery',
    )
    print(
        'Llama stats use LLM gold labels; BERT stats use BERT gold labels; recovery is computed on shared rows only.',
    )
    print('=' * 120)
    for (ctx, lvl), sub_lp in stats_df.groupby(['context', 'prompt_level'], sort=False):
        print('\n' + '-' * 120)
        print(f'{ctx.upper()} | {lvl}')
        print('-' * 120)
        for cat, sub_cat in sub_lp.groupby('category_code', sort=False):
            cat_name = sub_cat['category_name'].iloc[0]
            print(f'\n{cat} {cat_name}')
            total_row = sub_cat[sub_cat['subcategory'] == CATEGORY_TOTAL_LABEL]
            sub_rows = sub_cat[sub_cat['subcategory'] != CATEGORY_TOTAL_LABEL]
            if not total_row.empty:
                r = total_row.iloc[0]
                llama_part = _fmt_side(
                    'Llama (LLM gold)',
                    r.get('llm_recognized_tp'),
                    r.get('llm_support_gold'),
                    r.get('llm_recall'),
                    r.get('llm_missed_fn'),
                )
                bert_part = _fmt_side(
                    'BERT (BERT gold)',
                    r.get('bert_recognized_tp'),
                    r.get('bert_support_gold'),
                    r.get('bert_recall'),
                    r.get('bert_missed_fn'),
                )
                rec_part = _fmt_recovery(
                    'Recovered from BERT-missed (shared rows)',
                    r.get('llama_recovered_from_bert_missed'),
                    r.get('bert_missed_support_shared'),
                    r.get('recovery_rate'),
                )
                print(f'  TOTAL: {llama_part} | {bert_part} | {rec_part}')
            if not PRINT_ALL_SUBCATS:
                sub_rows = sub_rows.head(TOP_SUBCATS_PER_CATEGORY)
            if sub_rows.empty:
                print('  (no subcategory rows)')
                continue
            for _, r in sub_rows.iterrows():
                llama_support = r.get('llm_support_gold')
                llama_tp = r.get('llm_recognized_tp')
                llama_missed = r.get('llm_missed_fn')
                llama_recall = r.get('llm_recall')
                bert_support = r.get('bert_support_gold')
                bert_tp = r.get('bert_recognized_tp')
                bert_missed = r.get('bert_missed_fn')
                bert_recall = r.get('bert_recall')
                rec_support = r.get('bert_missed_support_shared')
                rec_tp = r.get('llama_recovered_from_bert_missed')
                rec_rate = r.get('recovery_rate')
                if is_na(llama_support) or is_na(llama_tp):
                    llama_txt = 'Llama n/a'
                else:
                    llama_txt = f'Llama {int(llama_tp)}/{int(llama_support)} (r={float(llama_recall):.3f}), missed={int(llama_missed)}'
                if is_na(bert_support) or is_na(bert_tp):
                    bert_txt = 'BERT n/a'
                else:
                    bert_txt = f'BERT {int(bert_tp)}/{int(bert_support)} (r={float(bert_recall):.3f}), missed={int(bert_missed)}'
                if is_na(rec_support) or is_na(rec_tp):
                    rec_txt = 'recov=n/a'
                else:
                    rec_txt = f'recov={int(rec_tp)}/{int(rec_support)} ({float(rec_rate):.3f})'
                print(f"  - {r['subcategory']}: {llama_txt} | {bert_txt} | {rec_txt}")


def print_control_summary_llm(df_level: pd.DataFrame, context_label: str, level: str) -> None:
    n_rows = len(df_level)
    gold_any_ref = int(df_level['_gold_set'].apply(lambda s: len(s) > 0).sum())
    pred_any_ref = int(df_level['_pred_set'].apply(lambda s: len(s) > 0).sum())
    print(
        f'  [{context_label} | {level}] rows={n_rows:,} | gold_any_ref={gold_any_ref:,} | pred_any_ref={pred_any_ref:,}',
    )


def print_control_summary_bert(df_bert: pd.DataFrame) -> None:
    n_rows = len(df_bert)
    gold_any_ref = int(df_bert['_bert_gold_set'].apply(lambda s: len(s) > 0).sum())
    pred_any_ref = int(df_bert['_bert_pred_set'].apply(lambda s: len(s) > 0).sum())
    print('\nBERT sentence-level baseline loaded successfully.')
    print(f'  rows={n_rows:,} | gold_any_ref={gold_any_ref:,} | pred_any_ref={pred_any_ref:,}')
    cat_bits = []
    for cat in CATEGORIES:
        g = int(df_bert['_bert_gold_set'].apply(lambda s: cat in s).sum())
        p = int(df_bert['_bert_pred_set'].apply(lambda s: cat in s).sum())
        cat_bits.append(f'{cat}: gold={g}, pred={p}')
    print('  category counts (gold vs pred):')
    print('  ' + ' | '.join(cat_bits))


def main():
    os.makedirs(EXPORT_DIR, exist_ok=True)
    print('Building subcategory map from letters pickle (this may take a moment)...')
    subcat_map = build_subcategory_map()
    print(f'Subcategory map built: {len(subcat_map):,} (letter_id, sentence_id, category) keys')
    print('\nLoading BERT sentence-level predictions for recovery analysis...')
    bert_df = None
    try:
        bert_df = load_bert_sentence_df(BERT_SENTENCE_CSV)
        print_control_summary_bert(bert_df)
    except Exception as e:
        print(
            f'\nWARNING: Skipping BERT baseline / recovery analysis (BERT load failed): {e}\nThe LLM-only subcategory recognition export will still run.',
        )
        bert_df = None
    all_stats = []
    all_recovered_instances = []
    for context_label, context_dir in CONTEXT_DIRS.items():
        if not os.path.isdir(context_dir):
            print(f'\nSkipping missing directory: {context_dir}')
            continue
        print(f'\nProcessing context: {context_label} -> {context_dir}')
        for lvl in LEVELS:
            path = os.path.join(context_dir, CSV_PATTERN.format(level=lvl))
            if not os.path.isfile(path):
                raise FileNotFoundError(f'Missing expected file: {path}')
            df_level = build_llm_level_df(path, context_label=context_label, level=lvl)
            print_control_summary_llm(df_level, context_label=context_label, level=lvl)
            stats_df_level, recovered_df_level = build_subcategory_stats_for_level(
                df_llm_level=df_level,
                context_label=context_label,
                level=lvl,
                subcat_map=subcat_map,
                df_bert=bert_df,
            )
            if stats_df_level is not None and (not stats_df_level.empty):
                all_stats.append(stats_df_level)
            if recovered_df_level is not None and (not recovered_df_level.empty):
                all_recovered_instances.append(recovered_df_level)
    if not all_stats:
        print('\nNo stats computed.')
        return
    stats_df = pd.concat(all_stats, ignore_index=True)
    stats_df = stats_df.sort_values(
        [
            'context_order',
            'prompt_order',
            'category_code',
            'is_total_row',
            'llm_support_gold',
            'bert_support_gold',
            'bert_missed_support_shared',
            'subcategory',
        ],
        ascending=[True, True, True, False, False, False, False, True],
        na_position='last',
    ).reset_index(
        drop=True,
    )
    for c in ['llm_recall', 'bert_recall', 'recovery_rate']:
        if c in stats_df.columns:
            stats_df[c] = pd.to_numeric(stats_df[c], errors='coerce').round(4)
    print_console_summary(stats_df)
    stats_df.to_csv(OUT_STATS_CSV, index=False)
    print(f'\nSaved subcategory recognition CSV: {OUT_STATS_CSV} (rows={len(stats_df):,})')
    if all_recovered_instances:
        recovered_all = pd.concat(all_recovered_instances, ignore_index=True)
        recovered_all = recovered_all.sort_values(
            [
                'context_order',
                'prompt_order',
                'category_code',
                'subcategory',
                COL_LETTER_ID,
                COL_SENTENCE_ID,
            ],
            ascending=[True, True, True, True, True, True],
        ).reset_index(
            drop=True,
        )
        recovered_all.to_csv(OUT_RECOVERED_INSTANCES_CSV, index=False)
        print(
            f'Saved recovered-instance CSV: {OUT_RECOVERED_INSTANCES_CSV} (rows={len(recovered_all):,})',
        )
    else:
        print(
            'No exact BERT-missed / Llama-recovered instances found (or BERT analysis was skipped).',
        )
if __name__ == '__main__':
    main()


## Label-token confidence

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from typing import List, Optional, Tuple
import pandas as pd

BASE_DIR = Path(str(paths.PROMPT_RESULTS / 'sentence'))
CONTEXT_DIRS = {'nocontext': BASE_DIR / 'nocontext', 'context_4': BASE_DIR / 'context_4'}
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
FILE_PATTERN = 'llm_results_sentence_multilabel_{level}.csv'
SLOT_JSON_COL_CANDIDATES = ['slot_logprob_json', 'slot_probs_json', 'slot_logprobs_json', 'slot_log_probs_json']
ID_COLS = ['letter_id', 'sentence_id']
CAT_NAME = {
    'A': 'Inspirations and Influences',
    'B': 'Writing Process and Creative Insight',
    'C': 'Elements of Fictional Craft',
    'D': 'Editorial and Publishing Processes',
    'E': 'Quality and Evaluation',
    'F': 'Financial and Legal Aspects',
    'G': 'Gratitude and Acknowledgement',
    'H': 'Marketing and Promotion',
    'I': 'Impact and Reception',
    'J': 'No reference to Dickens’s works present',
}
from prompting.analysis_utils import read_file


def pick_slot_json_col(df: pd.DataFrame, path: Path) -> str:
    for c in SLOT_JSON_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(
        f'{path.name}: none of the expected JSON columns found. Tried: {SLOT_JSON_COL_CANDIDATES}',
    )


def safe_load_json(x) -> Optional[List[dict]]:
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return None
    try:
        data = json.loads(s)
        return data if isinstance(data, list) else None
    except Exception:
        return None


def extract_slot_predictions(slot_list: List[dict]) -> List[Tuple[str, float]]:
    out: List[Tuple[str, float]] = []
    for d in slot_list:
        tok = d.get('token', None)
        if tok is None:
            tok = d.get('label', None)
        if tok is None:
            continue
        tok = str(tok).strip().upper()
        if tok not in CAT_NAME:
            continue
        try:
            prob = float(d.get('prob', None))
        except Exception:
            continue
        out.append((tok, prob))
    return out


def summarise_confidence(long_df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame(
            columns=group_cols + ['category', 'category_name', 'n', 'mean', 'std', 'min', 'max'],
        )
    stats = long_df.groupby(group_cols + ['category'])['confidence'].agg(
        n='count',
        mean='mean',
        std='std',
        min='min',
        max='max',
    ).reset_index()
    stats['category_name'] = stats['category'].map(CAT_NAME)
    stats = stats[group_cols + ['category', 'category_name', 'n', 'mean', 'std', 'min', 'max']]
    return stats


def print_section(title: str) -> None:
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))


def print_table_sorted(df: pd.DataFrame, per_level: bool=False) -> None:
    if df.empty:
        print('(no data)')
        return
    show = df.copy()
    for c in ['mean', 'std', 'min', 'max']:
        if c in show.columns:
            show[c] = show[c].astype(float).round(4)
    if per_level and 'level' in show.columns:
        show = show.sort_values(['level', 'mean', 'category'], ascending=[True, False, True])
    else:
        show = show.sort_values(['mean', 'category'], ascending=[False, True])
    print(show.to_string(index=False))


def main() -> None:
    all_rows: List[dict] = []
    for ctx, ctx_dir in CONTEXT_DIRS.items():
        if not ctx_dir.exists():
            print(f' Missing directory, skipping: {ctx_dir}')
            continue
        for lvl in LEVELS:
            path = ctx_dir / FILE_PATTERN.format(level=lvl)
            if not path.exists():
                print(f' Missing file, skipping: {path.name} ({ctx})')
                continue
            df = read_file(path)
            slot_col = pick_slot_json_col(df, path)
            for c in ID_COLS:
                if c not in df.columns:
                    df[c] = None
            for _, row in df.iterrows():
                slot_list = safe_load_json(row.get(slot_col))
                if not slot_list:
                    continue
                preds = extract_slot_predictions(slot_list)
                if not preds:
                    continue
                for tok, prob in preds:
                    all_rows.append(
                        {
                            'context': ctx,
                            'level': lvl,
                            'letter_id': row.get('letter_id'),
                            'sentence_id': row.get('sentence_id'),
                            'category': tok,
                            'confidence': prob,
                        },
                    )
    long_df = pd.DataFrame(all_rows)
    if long_df.empty:
        print(' No confidence data extracted. Check JSON column name/content.')
        return
    per_ctx_level = summarise_confidence(long_df, group_cols=['context', 'level'])
    pooled_ctx = summarise_confidence(long_df, group_cols=['context'])
    pooled_all = summarise_confidence(long_df, group_cols=[])
    print_section('NOCONTEXT — PER PROMPT LEVEL (sorted by mean within each level)')
    print_table_sorted(per_ctx_level[per_ctx_level['context'] == 'nocontext'], per_level=True)
    print_section('NOCONTEXT — POOLED ACROSS LEVELS (sorted by mean)')
    print_table_sorted(pooled_ctx[pooled_ctx['context'] == 'nocontext'], per_level=False)
    print_section('CONTEXT (n=4) — PER PROMPT LEVEL (sorted by mean within each level)')
    print_table_sorted(per_ctx_level[per_ctx_level['context'] == 'context_4'], per_level=True)
    print_section('CONTEXT (n=4) — POOLED ACROSS LEVELS (sorted by mean)')
    print_table_sorted(pooled_ctx[pooled_ctx['context'] == 'context_4'], per_level=False)
    print_section('ALL CONTEXTS — POOLED (sorted by mean)')
    pooled_all = pooled_all.copy()
    pooled_all.insert(0, 'context', 'ALL')
    pooled_all.insert(1, 'level', 'POOLED')
    print_table_sorted(pooled_all.drop(columns=['context', 'level']), per_level=False)
if __name__ == '__main__':
    main()


## Context effects

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROMPT_LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
CONTEXT_SIZES = [0, 4]
CATEGORIES = ['Writing Process & Creative Insight', 'Financial & Legal Aspects']
OUT_DIR = Path(str(paths.FIGURES_DIR))
OUT_DIR.mkdir(parents=True, exist_ok=True)
COLOR_CTX0 = '#B3B3B3'
COLOR_CTX4 = '#4D4D4D'


def make_llama_data() -> pd.DataFrame:
    scores = pd.read_csv(paths.PROMPT_RESULTS / 'summaries' / 'category_scores.csv')
    llama_data = {}
    for category, group in scores.groupby('Category'):
        name = category.replace(' and ', ' & ')
        llama_data[name] = {(0 if row.Context == 'nocontext' else 4, row.Prompt): row.F1 for row in group.itertuples()}
    rows = []
    for cat in CATEGORIES:
        vals = llama_data[cat]
        for ctx in CONTEXT_SIZES:
            for p in PROMPT_LEVELS:
                rows.append(
                    {'prompt': p, 'context_n': ctx, 'category': cat, 'f1': float(vals[ctx, p])},
                )
    return pd.DataFrame(rows)


def plot_two_panels(df: pd.DataFrame) -> None:
    plt.rcParams.update(
        {
            'font.size': 13,
            'axes.titlesize': 14,
            'axes.labelsize': 13,
            'xtick.labelsize': 12,
            'ytick.labelsize': 12,
            'pdf.fonttype': 42,
            'ps.fonttype': 42,
        },
    )
    fig = plt.figure(figsize=(13.6, 5.2))
    gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[18, 2], hspace=0.15, wspace=0.12)
    ax_left = fig.add_subplot(gs[0, 0])
    ax_right = fig.add_subplot(gs[0, 1], sharey=ax_left)
    leg_ax = fig.add_subplot(gs[1, :])
    leg_ax.axis('off')
    axes = [ax_left, ax_right]
    x = np.arange(len(PROMPT_LEVELS))
    bar_w = 0.38
    for ax, cat in zip(axes, CATEGORIES):
        n0 = df[(df['category'] == cat) & (df['context_n'] == 0)].set_index('prompt').reindex(
            PROMPT_LEVELS,
        )['f1'].values
        n4 = df[(df['category'] == cat) & (df['context_n'] == 4)].set_index('prompt').reindex(
            PROMPT_LEVELS,
        )['f1'].values
        b0 = ax.bar(
            x - bar_w / 2,
            n0,
            width=bar_w,
            color=COLOR_CTX0,
            edgecolor='black',
            label='Llama (context window n=0)',
        )
        b4 = ax.bar(
            x + bar_w / 2,
            n4,
            width=bar_w,
            color=COLOR_CTX4,
            edgecolor='black',
            label='Llama (context window n=4)',
        )
        ax.set_title(cat)
        ax.set_xticks(x)
        ax.set_xticklabels(PROMPT_LEVELS)
        ax.set_xlabel('Prompt level')
        ax.set_ylim(0, 1)
        ax.grid(True, axis='y', alpha=0.25)
    ax_left.set_ylabel('F1')
    handles, labels = ax_left.get_legend_handles_labels()
    leg_ax.legend(handles, labels, loc='center', ncol=2, frameon=True)
    out = OUT_DIR / 'prompt_dynamics_bars_writing_financial.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Saved: {out.resolve()}')


def main() -> None:
    df = make_llama_data()
    plot_two_panels(df)
if __name__ == '__main__':
    main()
